# Compare Technology Cost Assumptions

**About this notebook:** this is a PyPSA-Earth comparison. It does not update the agreed list of existing CEB assets.

**Purpose:** compare technology costs from different sources and years before deciding whether to rerun a possible-future-system model. The current example compares `pypsa-earth/data/costs.csv` with archived Danish Energy Agency (DEA) estimates for 2030.

**Before running:** use the repository `.venv` kernel. Check the currency year, units and technology definition for each source.

**What you need to do:** decide which source and year are suitable. Make any final change in the saved PyPSA-Earth input files, not only in this notebook. Changing costs requires the affected model runs to be repeated.

**Common adjustments:** you can compare other cost sources or technologies and report all values in another year's currency. State the inflation or currency conversion used.

In [ ]:
import pandas as pd
import yaml
from pathlib import Path
from IPython.display import display, HTML

## Current model cost assumptions

In [ ]:
costs_csv = pd.read_csv("../../pypsa-earth/data/costs.csv")

# Pivot to get investment, FOM, VOM, lifetime, efficiency, fuel per technology
costs_wide = costs_csv.pivot_table(
    index="technology", columns="parameter", values="value", aggfunc="first"
)

# Focus on the key technologies we want to compare
key_techs = [
    "solar", "onwind", "offwind", "nuclear", "OCGT", "CCGT",
    "electrolysis", "fuel cell", "hydrogen storage tank",
    "battery storage", "battery inverter",
    "ammonia synthesis", "ammonia storage",
    "CCGT H2", "CCGT NH3", "NH3 pipeline", "H2 pipeline",
]
available_techs = [t for t in key_techs if t in costs_wide.index]

cols_of_interest = ["investment", "FOM", "VOM", "lifetime", "efficiency", "fuel"]
cols_available = [c for c in cols_of_interest if c in costs_wide.columns]

current_costs = costs_wide.loc[available_techs, cols_available].copy()

# Get sources/currency year from raw CSV
inv_rows = costs_csv[
    (costs_csv["technology"].isin(available_techs)) & (costs_csv["parameter"] == "investment")
][["technology", "source", "currency_year"]].set_index("technology")

current_costs["source"] = inv_rows["source"]
current_costs["currency_year"] = inv_rows["currency_year"]

print("=== Current Model Costs (costs.csv) ===")
print("Note: investment is in EUR/kW (generators/links) or EUR/kWh (stores)")
print("All monetary values are in the currency_year shown (mostly 2013 EUR)\n")
display(current_costs.style.format(precision=2, na_rep="—"))

## Alternative DEA estimates for 2030

In [ ]:
with open("../archive/tech_config_ammonia_plant_2030_dea.yaml") as f:
    dea_config = yaml.safe_load(f)

# Extract DEA costs into a comparable DataFrame
dea_rows = []
for tech_name, tech_data in dea_config["techs"].items():
    ctype = tech_data.get("component_type", "")
    if ctype == "generator" or ctype == "link":
        cost_key = "overnight_cost_per_mw"
        unit = "EUR/kW"
        divisor = 1000  # Convert EUR/MW to EUR/kW
    elif ctype == "store":
        cost_key = "overnight_cost_per_mwh"
        unit = "EUR/kWh"
        divisor = 1000
    else:
        continue
    
    # Skip penalty/placeholder components
    if "penalty" in tech_name:
        continue
    
    inv = tech_data.get(cost_key, 0) / divisor
    dea_rows.append({
        "dea_tech": tech_name,
        "component_type": ctype,
        "investment_2020EUR": inv,
        "unit": unit,
        "lifetime": tech_data.get("lifetime_years"),
        "FOM_pct": tech_data.get("fixed_om_fraction", 0) * 100,
        "efficiency": tech_data.get("overall_efficiency"),
        "source": tech_data.get("source_note", ""),
    })

# Add DEA 2030 technologies not in the archived YAML
# CCGT: DEA "Gas turb. CC, steam extract." — 882.6 EUR/kW, eff 0.61, lifetime 25y, FOM 3.4%
dea_rows.append({
    "dea_tech": "ccgt_gas",
    "component_type": "generator",
    "investment_2020EUR": 882.6,  # EUR/kW (already in kW basis)
    "unit": "EUR/kW",
    "lifetime": 25,
    "FOM_pct": 3.4,
    "efficiency": 0.61,
    "source": "DEA 2030 Gas turb. CC, steam extract.",
})

dea_costs = pd.DataFrame(dea_rows).set_index("dea_tech")

print("=== DEA 2030 Costs (from archived tech config + manual entries, 2020 EUR) ===")
display(dea_costs.style.format(precision=2, na_rep="—"))

## Compare current and alternative costs

The code matches equivalent technology names across the two sources.

To express the DEA values in 2013 euros, this example removes estimated euro-area inflation between 2013 and 2020. It uses a factor of `1 / 1.084 = 0.9225`. Review this assumption before using the converted values in published work.

In [ ]:
# Mapping: DEA tech name → costs.csv tech name
dea_to_csv_map = {
    "solar": "solar",
    "onshore_wind": "onwind",
    "offshore_wind_fixed": "offwind",
    "ccgt_gas": "CCGT",
    # No DEA nuclear — DIW only
    "electrolysis": "electrolysis",
    "hydrogen_fuel_cell": "fuel cell",
    "ammonia_synthesis": "ammonia synthesis",
    "battery_pcs_charge": "battery inverter",
    "battery_storage": "battery storage",
    "compressed_hydrogen_store": "hydrogen storage tank",
    "ammonia": "ammonia storage",
}

# Eurozone HICP cumulative inflation 2013→2020: ~8.4%
# Source: ECB Statistical Data Warehouse, HICP overall index
DEFLATOR_2020_TO_2013 = 1 / 1.084  # ≈ 0.9225

comparison_rows = []
for dea_name, csv_name in dea_to_csv_map.items():
    if dea_name not in dea_costs.index:
        continue
    dea_row = dea_costs.loc[dea_name]
    
    csv_inv = current_costs.loc[csv_name, "investment"] if csv_name in current_costs.index else None
    csv_lifetime = current_costs.loc[csv_name, "lifetime"] if csv_name in current_costs.index else None
    csv_source = current_costs.loc[csv_name, "source"] if csv_name in current_costs.index else "—"
    
    dea_inv_2020 = dea_row["investment_2020EUR"]
    dea_inv_2013 = dea_inv_2020 * DEFLATOR_2020_TO_2013
    
    pct_diff_vs_2020 = (
        (csv_inv - dea_inv_2020) / dea_inv_2020 * 100 if csv_inv and dea_inv_2020 else None
    )
    pct_diff_vs_2013 = (
        (csv_inv - dea_inv_2013) / dea_inv_2013 * 100 if csv_inv and dea_inv_2013 else None
    )
    
    comparison_rows.append({
        "technology": csv_name,
        "dea_tech": dea_name,
        "unit": dea_row["unit"],
        "current_csv (2013 EUR)": csv_inv,
        "DEA 2030 (2020 EUR)": dea_inv_2020,
        "DEA 2030 (→2013 EUR)": dea_inv_2013,
        "csv vs DEA_2020 (%)": pct_diff_vs_2020,
        "csv vs DEA_2013 (%)": pct_diff_vs_2013,
        "csv_source": csv_source,
    })

comp_df = pd.DataFrame(comparison_rows).set_index("technology")

print("=== Investment Cost Comparison: Current costs.csv vs DEA 2030 ===")
print("Positive % = costs.csv is MORE expensive than DEA")
print("Negative % = costs.csv is CHEAPER than DEA\n")
display(
    comp_df.style.format({
        "current_csv (2013 EUR)": "{:,.1f}",
        "DEA 2030 (2020 EUR)": "{:,.1f}",
        "DEA 2030 (→2013 EUR)": "{:,.1f}",
        "csv vs DEA_2020 (%)": "{:+.1f}%",
        "csv vs DEA_2013 (%)": "{:+.1f}%",
    }, na_rep="—")
    .map(
        lambda v: "color: green" if isinstance(v, str) and v.startswith("+") else
                  ("color: red" if isinstance(v, str) and v.startswith("-") else ""),
        subset=["csv vs DEA_2020 (%)", "csv vs DEA_2013 (%)"]
    )
)

## Expected working life of each technology

In [ ]:
lifetime_rows = []
for dea_name, csv_name in dea_to_csv_map.items():
    if dea_name not in dea_costs.index:
        continue
    dea_lt = dea_costs.loc[dea_name, "lifetime"]
    csv_lt = current_costs.loc[csv_name, "lifetime"] if csv_name in current_costs.index else None
    
    lifetime_rows.append({
        "technology": csv_name,
        "csv_lifetime_yr": csv_lt,
        "dea_lifetime_yr": dea_lt,
        "difference_yr": (csv_lt - dea_lt) if csv_lt and dea_lt else None,
    })

lt_df = pd.DataFrame(lifetime_rows).set_index("technology")
print("=== Lifetime Comparison ===")
display(lt_df)

## Technologies without a direct DEA match

These technologies in `costs.csv` do not have a clear equivalent in the archived DEA file:

In [ ]:
techs_with_dea = set(dea_to_csv_map.values())
techs_without_dea = [t for t in available_techs if t not in techs_with_dea]

if techs_without_dea:
    no_dea = current_costs.loc[techs_without_dea, ["investment", "lifetime", "source", "currency_year"]]
    print("=== Technologies in costs.csv with NO DEA 2030 equivalent ===")
    display(no_dea)
else:
    print("All key technologies have DEA equivalents.")

## Main points to review

- Solar and wind costs in `costs.csv` come from older sources and are mostly reported in 2013 euros.
- The nuclear cost in the current file is below many more recent estimates.
- Electrolysis and ammonia costs already use DEA 2030 sources.
- Battery cost assumptions may differ substantially between sources.
- The current fuel-cell estimate is much lower than the DEA estimate, although fuel cells are not enabled in the current runs.

Before final runs:

1. Choose one clear cost source and reporting year.
2. Check whether the offshore-wind estimate matches the connection type being modelled.
3. Review the nuclear estimate against a more recent source.
4. Repeat all affected runs after changing a cost.
5. State the currency year whenever costs are reported.

## Running cost and carbon policy

**Short-run marginal cost (SRMC)** means the cost of producing one additional MWh from a power station that has already been built. It includes fuel, variable operating cost and any carbon cost. The model normally uses the cheapest available stations first.

`SRMC = fuel cost / efficiency + variable operating cost + carbon cost`

PyPSA-Earth can represent carbon policy in two ways:

| Config label | Plain-language meaning | Current use |
|---|---|---|
| `Co2L` | Limit total annual CO2. The model calculates how costly that limit is. | Used for limited, near-zero and effectively unlimited cases. |
| `Ep` | Add a fixed carbon price directly to power-station running costs. | Not used in the current main cases. |

When the model calculates the cost of tightening an emissions limit, this is often called the **shadow price**. It is the extra system cost associated with allowing one less unit of emissions.

In [ ]:
# === SRMC Merit Order: Comparative table with carbon break-even ===
import pandas as pd

data = {
    "Carrier":           ["Lignite", "Coal",  "CCGT",  "OCGT",  "Oil",   "Nuclear"],
    "Fuel (EUR/MWhth)":  [6.0,       15.0,    40.0,    40.0,    54.2,    3.3],
    "Efficiency":        [0.447,     0.464,   0.580,   0.410,   0.393,   0.337],
    "CO2 (t/MWhth)":     [0.336,     0.336,   0.198,   0.198,   0.248,   0.0],
    "VOM (EUR/MWhel)":   [3.0,       3.5,     3.3,     3.3,     0.0,     0.0],
}
df = pd.DataFrame(data).set_index("Carrier")

# Derived columns
df["Fuel (EUR/MWhel)"] = df["Fuel (EUR/MWhth)"] / df["Efficiency"]
df["CO2 (t/MWhel)"]    = df["CO2 (t/MWhth)"] / df["Efficiency"]
df["SRMC no CO2"]       = df["Fuel (EUR/MWhel)"] + df["VOM (EUR/MWhel)"]

# Carbon price at which each fuel's total SRMC equals CCGT's total SRMC
# SRMC_i + CO2_el_i * Pc = SRMC_ccgt + CO2_el_ccgt * Pc
# => Pc = (SRMC_ccgt - SRMC_i) / (CO2_el_i - CO2_el_ccgt)
ccgt_srmc   = df.loc["CCGT", "SRMC no CO2"]
ccgt_co2_el = df.loc["CCGT", "CO2 (t/MWhel)"]
df["CO2 price to match CCGT (EUR/t)"] = (
    (ccgt_srmc - df["SRMC no CO2"]) / (df["CO2 (t/MWhel)"] - ccgt_co2_el)
).round(0)
df.loc["CCGT",    "CO2 price to match CCGT (EUR/t)"] = float("nan")
df.loc["Nuclear", "CO2 price to match CCGT (EUR/t)"] = float("nan")

# Display
cols = [
    "Fuel (EUR/MWhth)", "Efficiency", "CO2 (t/MWhth)",
    "Fuel (EUR/MWhel)", "VOM (EUR/MWhel)", "SRMC no CO2",
    "CO2 (t/MWhel)", "CO2 price to match CCGT (EUR/t)",
]
display(df[cols].round(1).style.set_caption(
    "SRMC Merit Order -- DEA 2030 fuel costs (2020 EUR), no carbon price"
).format(precision=1, na_rep="--"))

print()
print("Key: 'CO2 price to match CCGT' = carbon price at which each fuel's")
print("total SRMC (fuel + VOM + carbon) equals CCGT's total SRMC.")
print(f"\nCCGT SRMC (no CO2 price) = {ccgt_srmc:.1f} EUR/MWhel")
print("EU ETS price 2024-26: approx 60-70 EUR/tCO2 => coal & lignite uncompetitive in reality.")